# TalentDesk, Section 1 Lab (Exercise): Build a Recruiter Agent

A hands-on exercise built on the **base Anthropic SDK**, running **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**. This lab combines the two
foundational agent skills you just learned: building the smallest possible agent, one
tool and one `tool_use` to `tool_result` round-trip (Lab 1), then adding a **second tool**
and a **system prompt** and watching how the prompt and tool descriptions steer which tool
the agent picks (Lab 2). You will fill in four short `TODO` blocks; everything else is
provided. A small offline mock lets you test your wiring without a key, and a full solution
is included at the end.

## The real-world scenario

Priya, a recruiting coordinator, gets the same two kinds of question all day: *"where is
candidate C1 in the process?"* and *"move C1 to the next round."* A plain chat model can
write a tidy reply, but it cannot **know** which stage C1 is really at; that fact lives in
the applicant tracking system (the ATS), not in the model. And it certainly should not
**advance** a candidate on its own without checking the rules.

An **agent** closes both gaps: it is an LLM that reasons about a goal and then **acts** on
its environment through tools. Here the environment is the ATS, and the two tools are a
**stage lookup** and a **guarded advance** that only moves a candidate who has cleared
screening.

The question this lab answers: **what is the minimum machinery that turns a chat model into
a recruiter agent that can look a candidate up, decide, and take the right action, and how
do the prompt and tool descriptions steer that decision?**

## Objectives

- Name the three building blocks every agent needs: an **LLM** (the reasoning engine), a
  **tool** (a function it can call to act), and a **prompt** (the goal that steers it).
- Define tools with a **JSON schema** so the model knows how to call them.
- Wire a single **tool_use to tool_result** round-trip end to end.
- Add a **second tool** so the agent must **choose**, and a **system prompt** that gives it a
  role and a goal.
- Observe how the **system prompt** and the **tool descriptions** change the agent's decision
  on the same request, and see why a two-step request points to the agentic loop next.

## The outcome you should reach

By the end you will have a working recruiter agent that:

- exposes two tools with JSON schemas: a stage lookup and a guarded advance;
- completes a full round-trip (the model decides with `stop_reason == "tool_use"`, your code
  runs the tool, hands back a `tool_result`, and the model answers with `end_turn`);
- routes a status question to the lookup and an advance request to the advance tool;
- and changes its behaviour when you tighten the system prompt (check the stage before
  advancing).

Target time: **20 to 30 minutes.** Four small `TODO` blocks, clearly marked. The
pure-Python parts (tools, schemas, wiring) are testable offline with a built-in mock; the
routing experiments need a real key.

## How to run

Run top to bottom. The tools, schemas, and the round-trip wiring are pure Python and run
anywhere; an offline **mock client** lets you exercise the round-trip without a key. The
routing experiments call Claude and observe its choices, so paste a real key into
**Setup 2/3** to run them live; otherwise they print the outcome you would expect. Live model
choices are not perfectly deterministic, so treat each live run as an observation.

## 0. Setup

**This cell:** installs the packages. This lab uses only the **base Anthropic SDK**,
because an agent is just a Messages API call with a `tools` list plus a little code around
it.

In [ ]:
# ===== SETUP 1/3 - install the base SDK =====
%pip install -q anthropic python-dotenv

**This cell:** imports what we need, pins the model, and sets a `RUN_LIVE` switch so
live model calls fire only when a real key is present. Offline, the switch routes the agent
to a mock client instead, so you can still test your wiring.

In [ ]:
# ===== SETUP 2/3 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import json                                     # print tool payloads readably
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (wiring runs on the offline mock)")

**This cell:** builds the shared **TalentDesk world**: a tiny ATS. Each candidate has
a pipeline **stage** and a **cleared** flag (did they pass screening?). C1 is at interview and
cleared to advance; C2 is still in screening and not cleared. The tools read from this.

In [ ]:
# ===== SETUP 3/3 - the shared ATS data (the agent's environment) =====
CANDIDATES = {                                   # our tiny applicant tracking system
    "C1": {"stage": 3, "cleared": True},         #   at interview, cleared to advance
    "C2": {"stage": 2, "cleared": False},        #   in screening, not cleared yet
}
STAGE_NAMES = {1: "applied", 2: "screening", 3: "interview", 4: "offer"}   # code -> human word
print("candidates:", list(CANDIDATES), "| stages:", list(STAGE_NAMES.values()))

### What is an agent?

A **single-shot prompt** takes text in and gives text out; it cannot reach outside itself. A
**hardcoded pipeline** runs fixed steps in a fixed order with no decisions. An **agent** sits
between them: an LLM that **reasons** about a goal and **acts** through tools, deciding for
itself whether to answer directly or call a tool first.

Every agent is made of three parts:

- **LLM**, the reasoning engine that decides what to do next.
- **Tools**, functions it can call to act on the world (here, read a stage or advance a
  candidate).
- **Prompt**, the role, instructions, and goal that steer it (here, the system prompt plus the
  recruiter's question).

**This cell:** the **offline mock client**, provided. Offline, the agent talks to this
instead of the real API so you can test your round-trip wiring without a key. It inspects the
messages: if it sees a `tool_result` already, it finishes with `end_turn`; otherwise it asks
for the stage tool (`tool_use`). It always picks the stage tool, so it tests **wiring**, not
routing; real routing needs a live key.

In [ ]:
# ===== the offline mock client (provided) - tests wiring, not routing =====
class _Blk:                                        # a stand-in for an SDK content block
    def __init__(self, type, **kw):
        self.type = type
        for k, v in kw.items():
            setattr(self, k, v)

class _Resp:                                       # a stand-in for an SDK response
    def __init__(self, stop_reason, content):
        self.stop_reason, self.content = stop_reason, content

class _Msgs:
    def create(self, **kw):                        # mimics client.messages.create(...)
        msgs = kw.get("messages", [])              #   look at the conversation so far
        last = msgs[-1] if msgs else {}
        content = last.get("content") if isinstance(last, dict) else None
        saw_result = isinstance(content, list) and any(   # did we already hand back a tool_result?
            isinstance(b, dict) and b.get("type") == "tool_result" for b in content)
        if saw_result:                             #   yes -> the model would now answer
            return _Resp("end_turn", [_Blk("text",
                text="Candidate C1 is at the interview stage. (canned offline reply)")])
        return _Resp("tool_use", [_Blk("tool_use",   # no -> the model asks for the stage tool
            id="mock_call_1", name="get_candidate_stage", input={"candidate_id": "C1"})])

class MockClient:                                  # the object run_once will call when offline
    def __init__(self): self.messages = _Msgs()

CLIENT = anthropic.Anthropic() if RUN_LIVE else MockClient()   # real client live, mock offline
print("client in use:", "real Anthropic" if RUN_LIVE else "offline MockClient")

---

### 🎯 The build - two tools, a round-trip, and a steering prompt

**What you build:** two tool bodies and their JSON schemas, the code that handles one
`tool_use` to `tool_result` round-trip, and a system prompt that routes each request to the
right tool.

**Why it helps you build real solutions:** every agent framework, however large, is this
round-trip repeated, and *what the agent does* is controlled by the prompt and the tool
descriptions far more than by clever code.

**TODO 1 (about 5 minutes).** Complete the second tool body, `advance_candidate()`. The
stage lookup is provided. `advance_candidate` is the **guarded action**: it applies the
business rule before it acts. Rules, in order:

- unknown candidate id, return `"unknown candidate"`;
- candidate not `cleared`, return `"blocked: screening not cleared"`;
- already at the final stage (`stage == 4`, offer), return `"already at final stage (offer)"`;
- otherwise, move to the next stage and return `"advanced to " + STAGE_NAMES[new_stage]`.

In [ ]:
# ===== TODO 1 - the two tool bodies (plain Python) =====
def get_candidate_stage(candidate_id):            # tool 1 (provided): read the pipeline stage
    c = CANDIDATES.get(candidate_id)              #   find the candidate
    return STAGE_NAMES[c["stage"]] if c else "unknown candidate"   # stage word, or a miss

def advance_candidate(candidate_id):              # tool 2 (YOU write this): the guarded action
    c = CANDIDATES.get(candidate_id)              #   find the candidate
    # 👉 TODO 1a: if there is no such candidate, return "unknown candidate"
    # 👉 TODO 1b: if the candidate is not cleared, return "blocked: screening not cleared"
    # 👉 TODO 1c: if already at stage 4, return "already at final stage (offer)"
    # 👉 TODO 1d: otherwise move to the next stage and return
    #             "advanced to " + STAGE_NAMES[c["stage"] + 1]
    return "TODO: implement advance_candidate"    # replace this line

RUN_TOOL = {"get_candidate_stage": get_candidate_stage,   # name -> function, dispatch by name
            "advance_candidate": advance_candidate}
print("tool bodies ready:", list(RUN_TOOL))

**Self-check (offline).** These assertions run without a key and confirm your rule
logic. If any fails, revisit TODO 1. When they pass you will see `TODO 1 checks passed`.

In [ ]:
# ===== self-check for TODO 1 (runs offline) =====
assert get_candidate_stage("C1") == "interview"
assert get_candidate_stage("Z9") == "unknown candidate"
assert advance_candidate("C1") == "advanced to offer"          # cleared + at interview -> offer
assert advance_candidate("C2") == "blocked: screening not cleared"   # not cleared
assert advance_candidate("Z9") == "unknown candidate"          # no such candidate
print("TODO 1 checks passed")

**TODO 2 (about 5 minutes).** Complete the `TOOLS` list, the schemas the model reads to
decide when and how to call each tool. The `description` is the **routing signal**: it is how
the model tells a status question from an advance request, so write each one sharply. Both
tools take a single required string, `candidate_id`.

Fill in: the two `description` strings, and the shared `input_schema` (an object with one
required string property `candidate_id`).

In [ ]:
# ===== TODO 2 - the tool schemas the model reads =====
# 👉 TODO 2a: build the shared input schema: an object with one required string, candidate_id.
_arg = {
    # "type": "object",
    # "properties": {"candidate_id": {"type": "string"}},
    # "required": ["candidate_id"],
}

TOOLS = [
    {"name": "get_candidate_stage",
     # 👉 TODO 2b: a sharp description for status / "where is this candidate" questions
     "description": "",
     "input_schema": _arg},
    {"name": "advance_candidate",
     # 👉 TODO 2c: a sharp description for move / promote / advance-to-next-round requests
     "description": "",
     "input_schema": _arg},
]
print("tools defined:", [t["name"] for t in TOOLS])

**Self-check (offline).** Confirms the schema shape and that every tool name matches a
real function in `RUN_TOOL`.

In [ ]:
# ===== self-check for TODO 2 (runs offline) =====
assert _arg.get("type") == "object" and "candidate_id" in _arg.get("properties", {})
assert _arg.get("required") == ["candidate_id"]
assert [t["name"] for t in TOOLS] == list(RUN_TOOL)             # names match the dispatch table
assert all(len(t["description"]) > 10 for t in TOOLS)          # descriptions are actually written
print("TODO 2 checks passed")

**This cell:** the **single-shot baseline**, no tools attached (provided, live-only). We
ask the model a candidate's stage with no way to look it up, so it can only guess. This is the
gap the agent closes.

In [ ]:
# ===== single-shot: no tools, so the model cannot know (provided) =====
QUESTION = "What stage is candidate C1 at in the hiring process?"

if RUN_LIVE:
    _c = anthropic.Anthropic()
    r = _c.messages.create(model=MODEL, max_tokens=200,        # a plain call, NO tools
                           messages=[{"role": "user", "content": QUESTION}])
    print("".join(b.text for b in r.content if b.type == "text"))   # it can only guess
else:
    print("[offline] with no tool, the model can only guess C1's stage; the agent below")
    print("          looks it up for real. Set a key to see the guess.")

**TODO 3 (about 6 minutes).** Complete `run_once()`, the round-trip. The four steps are
marked. Two are done; you fill the middle:

- STEP 2: read the chosen tool call and **run** the matching Python function via `RUN_TOOL`.
- STEP 3: append the assistant turn, then hand the result back as a `tool_result` carrying the
  **exact** `tool_use_id` from the call.

The offline mock will drive this: first it returns a `tool_use` for the stage tool, then, once
it sees your `tool_result`, it finishes with `end_turn`. If you forget the `tool_result`, the
mock keeps asking, which is the signal that STEP 3 is missing.

In [ ]:
# ===== TODO 3 - one tool_use -> tool_result round-trip =====
def run_once(question, system, tools):            # take a goal, use a tool once, then answer
    messages = [{"role": "user", "content": question}]        # the prompt: role (system) + goal

    first = CLIENT.messages.create(model=MODEL, max_tokens=512,   # STEP 1: the model decides
                                   system=system, tools=tools, messages=messages)
    if first.stop_reason != "tool_use":           #   chose not to use a tool?
        return "".join(b.text for b in first.content if b.type == "text")   # just answer

    call = next(b for b in first.content if b.type == "tool_use")   # the chosen tool call
    print("  chosen tool:", call.name, call.input)

    # 👉 TODO 3a (STEP 2): run the chosen tool and capture its result
    #    result = RUN_TOOL[...](**...)
    result = None                                  # replace this line
    print("  tool result:", result)

    # 👉 TODO 3b (STEP 3): keep the assistant turn, then hand the result back
    messages.append({"role": "assistant", "content": first.content})   # (provided) keep the turn
    # messages.append({"role": "user", "content": [
    #     {"type": "tool_result", "tool_use_id": ..., "content": result}]})

    second = CLIENT.messages.create(model=MODEL, max_tokens=512,   # STEP 4: the model continues
                                    system=system, tools=tools, messages=messages)
    if second.stop_reason == "tool_use":           #   it wants ANOTHER tool (two-step request)
        nxt = next(b for b in second.content if b.type == "tool_use")
        print("  still wants:", nxt.name, "-> a LOOP would continue here (next section)")
        return "(incomplete: needs another step)"
    return "".join(b.text for b in second.content if b.type == "text")   # the final answer

**TODO 4 (about 4 minutes).** Write the base **system prompt**, the prompt building block
that gives the agent a role and a goal. It should:

- give the agent a role (a calm, concise recruiting coordinator);
- state the goal (resolve the recruiter's request using the tools);
- route explicitly: use **get_candidate_stage** for status questions and **advance_candidate**
  for advance requests.

A stricter variant, `SYSTEM_STRICT`, is provided for the experiments; you write `SYSTEM`.

In [ ]:
# ===== TODO 4 - the system prompt: role, goal, routing =====
# 👉 Write SYSTEM so it names the role, the goal, and both tools by name.
SYSTEM = ""   # replace with your system prompt

# provided: a stricter variant used in Experiment 2 (checks stage before advancing)
SYSTEM_STRICT = (
    "You are TalentDesk, a calm, concise recruiting coordinator. "
    "Before you ever advance a candidate, you MUST first check their stage with "
    "get_candidate_stage. Never call advance_candidate without checking the stage first."
)
print("SYSTEM set:", bool(SYSTEM.strip()))

**Self-check (offline).** A gentle check that your `SYSTEM` names the role and both
tools. It prints what it found rather than failing hard.

In [ ]:
# ===== self-check for TODO 4 (runs offline) =====
_s = SYSTEM.lower()
for label, needle in [("mentions get_candidate_stage", "get_candidate_stage"),
                      ("mentions advance_candidate", "advance_candidate"),
                      ("has a role word", "coordinator")]:
    print(("[ok] " if needle in _s else "[missing] ") + label)

**This cell:** the **wiring self-check** for your agent. Offline it runs on the mock and
should complete a full round-trip: a `tool_use` for the stage tool, your dispatch and
`tool_result`, then a canned `end_turn` reply. Live, it runs for real. Finish TODOs 1 to 4
first.

In [ ]:
# ===== run the agent once (mock offline, real live) =====
print("ANSWER:", run_once("What stage is candidate C1 at?", SYSTEM, TOOLS))

---

### Experiment 1 - routing by intent (live)

The same agent should send a **status** question to `get_candidate_stage` and an **advance**
request to `advance_candidate`. The tool descriptions you wrote in TODO 2 are doing the
routing.

In [ ]:
# ===== experiment 1: the same agent routes two intents =====
if RUN_LIVE:
    print("Q: status  ->"); print("A:", run_once("Where is candidate C1 in the process?", SYSTEM, TOOLS))
    print()
    print("Q: advance ->"); print("A:", run_once("Please advance candidate C1 to the next round.", SYSTEM, TOOLS))
else:
    print("[offline] expected live: the status question picks get_candidate_stage,")
    print("          the advance request picks advance_candidate. The mock always picks")
    print("          the stage tool, so this routing needs a real key to observe.")

### Experiment 2 - the system prompt changes the decision (live)

Swap in `SYSTEM_STRICT`, which says to check the stage **before** advancing. On the advance
request the agent now reaches for `get_candidate_stage` first. Same tools, same request: only
the prompt changed.

In [ ]:
# ===== experiment 2: a stricter system prompt changes what the agent does =====
if RUN_LIVE:
    print("with the strict prompt, the advance request starts by checking the stage:")
    print("A:", run_once("Advance candidate C2 to the next round.", SYSTEM_STRICT, TOOLS))
else:
    print("[offline] expected live: the agent now picks get_candidate_stage FIRST, then")
    print("          reports that the advance step still remains -> that needs a loop.")

### Experiment 3 - a two-step request shows why we need a loop (live)

A single request that needs two tools in order (check the stage, then advance) cannot finish in
one round-trip: `run_once` handles the first tool and then reports that another is still wanted.
That leftover step is exactly what the **agentic loop** in the next section carries to
completion.

In [ ]:
# ===== experiment 3: a two-step request stalls in one round-trip =====
if RUN_LIVE:
    print("A:", run_once("Check candidate C2's stage, then advance them.", SYSTEM_STRICT, TOOLS))
else:
    print("[offline] expected live: one round-trip does step 1 (the stage check) and then")
    print("          says another tool is still wanted. The loop keeps going until done.")

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| ask a plain prompt for live facts it cannot know | give it a tool and let it look the fact up |
| guess the tool from the reply text | branch on `stop_reason == "tool_use"`, not on prose |
| forget the `tool_use_id` on the result | echo the exact id so the model matches result to call |
| leave a tool description vague and hope | write a sharp description that says exactly when to use each tool |
| bury a must-do rule in code the model cannot see | put it in the system prompt where it steers the choice |
| assume one round-trip finishes any task | recognise multi-step requests and let a loop continue them |

### When is an agent the right choice?

An agent is not always the answer. It costs more (several model calls), adds latency, and is
less predictable than fixed code. Reach for the simplest thing that works:

| Situation | Better choice | Why |
|---|---|---|
| One fixed transformation of known input | a single prompt | cheapest, one call, deterministic |
| Fixed steps in a fixed order, no branching | a hardcoded pipeline | reliable, no per-step model call |
| Goal needs live data or genuine decisions | an agent | it reasons and calls tools as needed |

**Lesson:** an agent is an **LLM** plus **tools** plus a **prompt**, wired so the model
can decide to act and then use what it learns. The whole mechanism is one round-trip: decide
(`tool_use`), run the tool, return the result, answer (`end_turn`). With more than one tool, the
agent's behaviour is steered mostly by two inputs you control: the **system prompt** and the
**tool descriptions**. And once a task needs several tools in order, one round-trip is not
enough, which is the reason the next section builds the **loop**.

---

## Recap - the three blocks, one round-trip, and what steers the choice

| Building block / lever | In this lab | Course topic |
|---|---|---|
| LLM | the Sonnet call that decides | the reasoning engine (Lab 1) |
| Tools | `get_candidate_stage` and `advance_candidate` + JSON schemas | functions the model can call (Lab 1) |
| Round-trip | `tool_use` -> run tool -> `tool_result` -> `end_turn` | how an agent operates end to end (Lab 1) |
| System prompt | role, goal, and a must-do rule | steering behaviour (Lab 2) |
| Tool descriptions | sharp text that routes each intent | steering the choice (Lab 2) |
| Multi-step request | check the stage, then advance | one round-trip stalls -> motivates the loop (Lab 2) |

**Try it next:** add a third tool (for example, `reject_candidate`) and a routing line for it in
`SYSTEM`, then watch the agent pick among three. Or blur one tool description and see the routing
get shakier. Next section turns this single round-trip into the full **agentic loop**, which
carries the two-step request all the way to done.